

Test each pipeline step independently 

## 1. Imports & Load Config

In [1]:
import sys
from pathlib import Path
import openeo
import xarray as xr
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# Load the unified Hydra config WITHOUT @hydra.main (notebook-friendly)
PACKAGE_ROOT = Path(r"c:\Git_projects\openeo_mountains_snow\src\openeo_mountains_snow")
sys.path.insert(0, str(PACKAGE_ROOT.parent))

base_cfg = OmegaConf.load(PACKAGE_ROOT / "conf" / "config.yaml")
exp_cfg = OmegaConf.load(PACKAGE_ROOT / "conf" / "experiment" / "reconstruction.yaml")

# Merge experiment into the base config under "experiment" key
cfg = OmegaConf.merge(base_cfg, {"experiment": exp_cfg})
print(OmegaConf.to_yaml(cfg))

connection:
  endpoint: openeo.dataspace.copernicus.eu
sentinel2_l2a:
  collection: SENTINEL2_L2A
  scl_band: SCL
  cloud_values:
  - 8
  - 9
  - 3
  - 10
sentinel2_l1c:
  collection: SENTINEL2_L1C
  bands:
  - B02
  - B03
  - B04
  - B08
  - B11
  - sunZenithAngles
  - sunAzimuthAngles
water_mask:
  collection: ESA_WORLDCOVER_10M_2021_V2
  band: MAP
  water_values:
  - 80
modis:
  stac_url: https://stac.eurac.edu/collections/MOD10A1v61
agera5:
  stac_url: https://stac.openeo.vito.be/collections/agera5_daily
  bands:
  - 2m_temperature_mean
  - dewpoint_temperature_mean
  - solar_radiation_flux
  band_aliases:
  - temperature-mean
  - dewpoint-temperature
  - solar-radiation-flux
geopotential:
  stac_url: https://artifactory.vgt.vito.be/artifactory/auxdata-public/geopotential.json
dem:
  slope_stac_url: https://stac.openeo.vito.be/collections/DEM_slope_30m
  aspect_stac_url: https://stac.openeo.vito.be/collections/DEM_aspec_30m
processing:
  crs: 32632
  resolution: 20.0
  cloud_prob: 

## 2. Connect to openEO

In [2]:
eoconn = openeo.connect(cfg.connection.endpoint).authenticate_oidc()

Authenticated using refresh token.


## 3. Define a TINY test extent

Use a much smaller spatial/temporal extent than the real experiment so each `.execute()` completes in seconds, not hours. Override the config values here.

In [3]:
# ---- SMALL test extent (≈1km x 1km, 5 days) ----
# Adjust these to your study area. Keep it TINY for testing!
spatial_extent = {
    "west": 631800.0,
    "south": 5173000.0,
    "east": 632800.0,   # just 1 km
    "north": 5174000.0, # just 1 km
    "crs": "EPSG:32632",
}
temporal_extent = ["2023-06-01", "2023-06-05"]  # just 5 days

# For MODIS / AGERA steps later
modis_temporal_extent = ["2023-01-20", "2023-01-21"]
agera_temporal_extent = ["2023-07-01", "2023-07-03"]

proc = cfg.processing
recon = cfg.reconstruction
print(f"Test area: {spatial_extent}")
print(f"Test period: {temporal_extent}")

Test area: {'west': 631800.0, 'south': 5173000.0, 'east': 632800.0, 'north': 5174000.0, 'crs': 'EPSG:32632'}
Test period: ['2023-06-01', '2023-06-05']


## 4. Test: Load raw Sentinel-2 data

Simplest possible test — can we load S2 data and download it?

In [4]:
# Load raw SCL band — this is the simplest data load in the pipeline
scl = eoconn.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["SCL"],
    max_cloud_cover=proc.cloud_prob,
)



## 5. Test: Snow Cover Fraction (spectral indices + representative pixels)

This is the improved SCF from `snow_cover_fraction.py` — the whole point of the integration. Download and visualize.

In [ ]:
from openeo_mountains_snow.snow_cover_fraction import snow_cover_fraction_cube

snow = snow_cover_fraction_cube(
    aoi=spatial_extent,
    time_period=temporal_extent,
    c=eoconn,
    cfg=cfg,
).rename_labels(dimension="bands", target=["snow"])

# Download as NetCDF and inspect
snow.download("test_snow.nc")
ds = xr.open_dataset("test_snow.nc")
print(ds)

# Plot first timestep
ds["snow"].isel(t=0).plot(figsize=(6, 5), cmap="Blues")
plt.title("Snow Cover Fraction (spectral indices + representative pixels)")
plt.show()

## 7. Test: SCF Masks & Conditional Probabilities

Test `compute_scf_masks` — this combines the simple snow mask with MODIS SCF to produce conditional probability inputs.

In [ ]:
from openeo_mountains_snow.snowcoverarea_reconstruction.scf_processing import compute_scf_masks

all_masks, labels_scf = compute_scf_masks(eoconn, cfg, spatial_extent, temporal_extent)
print(f"✅ SCF mask labels: {labels_scf}")

# Download to verify
all_masks.download("test_scf_masks.nc")
ds_masks = xr.open_dataset("test_scf_masks.nc")
print(ds_masks)

## 8. Test: Downscale Temperature & Humidity

Test climate downscaling independently. This loads AGERA5, DEM, and geopotential.

In [ ]:
from openeo_mountains_snow.snowcoverarea_reconstruction.downscale_variables import (
    downscale_temperature_humidity, downscale_shortwave_radiation,
)

# Use a fixed date for time dimension
first_date = temporal_extent[0]

# DEM
dem = eoconn.load_collection("COPERNICUS_30", spatial_extent=spatial_extent)
if dem.metadata.has_temporal_dimension():
    dem = dem.reduce_dimension(dimension="t", reducer="max")
dem = dem.add_dimension(name="t", label=first_date, type="temporal")

# AGERA5
agera = eoconn.load_stac(
    cfg.agera5.stac_url,
    spatial_extent=spatial_extent,
    temporal_extent=agera_temporal_extent,
)
agera = agera.filter_bands(bands=list(cfg.agera5.bands))
agera = agera.rename_labels(dimension="bands", target=list(cfg.agera5.band_aliases))

# Geopotential
geopotential = eoconn.load_stac(
    cfg.geopotential.stac_url,
    spatial_extent=spatial_extent,
    bands=["geopotential"],
)
geopotential.metadata = geopotential.metadata.add_dimension(
    "t", label=first_date, type="temporal"
)

agera_downscaled = downscale_temperature_humidity(agera, dem, geopotential.max_time())

agera_downscaled.download("test_agera_downscaled.nc")
ds_agera = xr.open_dataset("test_agera_downscaled.nc")
print(ds_agera)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, var in enumerate(ds_agera.data_vars):
    if ds_agera[var].dims == ("t", "y", "x"):
        ds_agera[var].isel(t=0).plot(ax=axes[i])
        axes[i].set_title(var)
plt.tight_layout()
plt.show()

## 9. Test: Downscale Shortwave Radiation

In [ ]:
aspect = eoconn.load_stac(
    cfg.dem.aspect_stac_url, spatial_extent=spatial_extent
).reduce_dimension(dimension="t", reducer="mean")

slope = eoconn.load_stac(
    cfg.dem.slope_stac_url, spatial_extent=spatial_extent
).reduce_dimension(dimension="t", reducer="mean")

slope_aspect = aspect.merge_cubes(slope).rename_labels(
    dimension="bands", target=["aspect", "slope"]
)

shortwave_rad_cube = downscale_shortwave_radiation(agera, slope_aspect)

shortwave_rad_cube.download("test_shortwave.nc")
ds_sw = xr.open_dataset("test_shortwave.nc")
print(ds_sw)

## 10. Chain it all together

Once each step above works individually, chain them and run the full pipeline on the tiny extent.

In [ ]:
# If all steps above passed, run the full pipeline on the tiny extent
from openeo_mountains_snow.snowcoverarea_reconstruction.pipeline import run_reconstruction

# Override the experiment config to use our tiny test extent
test_cfg = OmegaConf.merge(cfg, {
    "experiment": {
        "aoi": spatial_extent,
        "temporal_extent": temporal_extent,
        "modis_temporal_extent": modis_temporal_extent,
        "agera_temporal_extent": agera_temporal_extent,
        "title_prefix": "test_baby_steps",
    }
})

# This will submit a batch job — use only when you're confident each step works!
# run_reconstruction(test_cfg, eoconn, spatial_extent)
print("🎉 Uncomment the line above when ready to run the full pipeline.")